# Using the `llmbenchmark` SDK in Jupyter

A step-by-step tour. Run the cells top to bottom.

| Step | You'll learn |
|---|---|
| 0 | Install the SDK and start the backend |
| 1 | `init()`: configure the SDK |
| 2 | Versioned inputs: `Prompt` and `Dataset` |
| 3 | Record a single prompt (with or without a system prompt) |
| 4 | Handle errors without stopping a benchmark loop |
| 5 | Benchmark two models over a dataset |
| 6 | Record a multi-turn conversation |
| 7 | Trace an agent: LLM calls, tool calls, sub-steps |
| 8 | Evaluate with llmbenchmark (managed evaluation) |
| 9 | Upload results you scored yourself (bring your own evaluation) |
| 10 | Version control: re-evaluate without overwriting |
| 11 | Compare runs and set a baseline |
| 12 | Read results back |
| 13 | What happens when the backend is down |

**No API keys?** The notebook still runs: it uses **mock** model responses, so you can learn the workflow for free.
Set `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` before starting Jupyter to use real models.

## Step 0: Setup

### 0a. Install the SDK

`%pip` (not `!pip`) installs into *this notebook's* Python. After installing or upgrading, **restart the kernel**.

In [ ]:
%pip install -q -e "..[eval,server]" anthropic openai

# From your private Artifactory index instead:
# %pip install -q --index-url https://<artifactory>/api/pypi/<repo>/simple "llmbenchmark[eval]==0.1.*" anthropic openai

### 0b. Start the backend

Normally you'd run this in a terminal from the repo root (see the README for per-team API keys):

```bash
LLMBENCHMARK_SERVER_ALLOW_ANONYMOUS=1 .venv/bin/uvicorn server.app:create_app --factory --port 8000
```

For convenience, the next cell starts it in the background if it isn't already running.

In [ ]:
import os, subprocess, sys, time
import httpx

BACKEND = "http://localhost:8000"

def backend_up():
    try:
        return httpx.get(f"{BACKEND}/healthz", timeout=1).status_code == 200
    except httpx.HTTPError:
        return False

server_process = None
if not backend_up():
    server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "server.app:create_app", "--factory", "--port", "8000"],
        cwd="..",                              # repo root, where server/ lives
        # Local development only: run without API keys. Real deployments use
        # LLMBENCHMARK_SERVER_KEYS_FILE (the server refuses to start with no keys).
        env={**os.environ, "LLMBENCHMARK_SERVER_ALLOW_ANONYMOUS": "1"},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(50):
        if backend_up():
            break
        time.sleep(0.2)

print("backend up:", backend_up(), "| API docs:", f"{BACKEND}/docs")

### 0c. API keys and mock mode

In [ ]:
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")   # DeepEval: no anonymous telemetry
os.environ.setdefault("DEEPEVAL_NO_INSPECT_PROMPT", "1")     # DeepEval: no interactive prompt after evals

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
USE_REAL_LLMS = HAS_ANTHROPIC and HAS_OPENAI

print("ANTHROPIC_API_KEY:", "set" if HAS_ANTHROPIC else "missing")
print("OPENAI_API_KEY:   ", "set" if HAS_OPENAI else "missing")
print("Mode:", "REAL model calls (costs tokens)" if USE_REAL_LLMS else "MOCK responses (free)")

## Step 1: Configure the SDK

`init()` is called once per notebook. Arguments you leave out fall back to environment variables
(`LLMBENCHMARK_ENDPOINT`, `LLMBENCHMARK_API_KEY`, `LLMBENCHMARK_PROJECT`, ...), then to defaults.
Calling it again (e.g. re-running the cell) is safe, even while a run is open.

A **project** groups related runs; a **run** is one experiment (one model + settings + prompt over a dataset).

In [ ]:
import llmbenchmark

config = llmbenchmark.init(
    endpoint=BACKEND,
    api_key=None,           # your team's key in a real deployment (or LLMBENCHMARK_API_KEY)
    project="tutorial",
)
print("llmbenchmark", llmbenchmark.__version__)
config                      # safe to display: the API key is never shown

## Step 2: Versioned inputs — `Prompt` and `Dataset`

Results are only comparable if you know exactly what produced them. A `Prompt` and a `Dataset` are
identified by a **hash of their content**: change one character and the hash changes. Runs record these
hashes, so "same prompt" and "same dataset" can be checked later.

Give dataset items stable `id`s: they are how two runs are compared question by question.

In [ ]:
SYSTEM = "Reply with only the answer: a few words, no full sentence, no trailing period."
prompt = llmbenchmark.Prompt("qa-short", system=SYSTEM, template="{input}", version="v1")

dataset = llmbenchmark.Dataset("qa-sample", [
    {"id": "france",    "input": "What is the capital of France?",                              "expected_output": "Paris"},
    {"id": "austen",    "input": "Who wrote 'Pride and Prejudice'?",                            "expected_output": "Jane Austen"},
    {"id": "gold",      "input": "What is the chemical symbol for gold?",                       "expected_output": "Au"},
    {"id": "continents","input": "How many continents are there?",                              "expected_output": "Seven"},
    {"id": "mars",      "input": "What planet is known as the Red Planet?",                     "expected_output": "Mars"},
    {"id": "boiling",   "input": "What is the boiling point of water at sea level in Celsius?", "expected_output": "100"},
    {"id": "monalisa",  "input": "Who painted the Mona Lisa?",                                  "expected_output": "Leonardo da Vinci"},
    {"id": "ocean",     "input": "What is the largest ocean on Earth?",                         "expected_output": "Pacific Ocean"},
], version="2026-11")

print(prompt)
print(dataset)

## Step 3: Record a single prompt

`track()` is a `with` block that **times** the call, reads **model, output text, token counts and tool calls**
from the response you assign to `t.response`, and records it when the block exits.

The input can be a plain prompt (`input=`), optionally with a system prompt (`system=`), or a dataset item (`item=`).

In [ ]:
import random

CLAUDE_MODEL = "claude-opus-5-5"
OPENAI_MODEL = "gpt-5"          # change to any chat model your OpenAI account has
GEN_PARAMS = {"max_tokens": 1024}

if USE_REAL_LLMS:
    import anthropic, openai
    claude_client = anthropic.Anthropic()
    openai_client = openai.OpenAI()


def ask_claude(question, system=SYSTEM):
    if not USE_REAL_LLMS:
        return mock_claude(question)
    return claude_client.messages.create(
        model=CLAUDE_MODEL, system=system, messages=[{"role": "user", "content": question}], **GEN_PARAMS,
    )


def ask_gpt(question, system=SYSTEM):
    if not USE_REAL_LLMS:
        return mock_gpt(question)
    return openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": question}],
    )


# ---- mock responses: same shape as the real providers' responses (as dicts) ----
MOCK_ANSWERS = {item.input: item.expected_output for item in dataset}
MOCK_GPT_MISTAKES = {
    "How many continents are there?": "7",                    # right answer, different format
    "What is the largest ocean on Earth?": "Atlantic Ocean",  # actually wrong
}

def mock_claude(question):
    time.sleep(random.uniform(0.02, 0.06))
    answer = MOCK_ANSWERS.get(question, "Paris")
    return {"type": "message", "id": "msg_mock", "model": CLAUDE_MODEL + "-mock", "stop_reason": "end_turn",
            "content": [{"type": "text", "text": answer}],
            "usage": {"input_tokens": 20 + len(question) // 4, "output_tokens": 3}}

def mock_gpt(question):
    time.sleep(random.uniform(0.02, 0.06))
    answer = MOCK_GPT_MISTAKES.get(question) or MOCK_ANSWERS.get(question, "Paris")
    return {"object": "chat.completion", "id": "chatcmpl_mock", "model": OPENAI_MODEL + "-mock",
            "choices": [{"message": {"content": answer}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": 25 + len(question) // 4, "completion_tokens": 4}}

In [ ]:
with llmbenchmark.start_run("single-prompt-demo", model=CLAUDE_MODEL, generation_params=GEN_PARAMS,
                            prompt=prompt) as run:
    with run.track(input="What is the capital of France?", system=SYSTEM, expected_output="Paris") as t:
        t.response = ask_claude("What is the capital of France?")
    with run.track(input="Say hello.") as t:          # no system prompt
        t.response = ask_claude("Say hello.", system=None)

r = run.records[0]
print("kind:   ", r.kind, "| system:", repr(r.system))
print("model:  ", r.model, "| output:", r.output)
print("tokens: ", r.input_tokens, "in /", r.output_tokens, "out | latency:", round(r.latency_ms), "ms")
print("second record system prompt:", run.records[1].system)

`start_run()` defaults to **managed mode** (you'll evaluate with llmbenchmark), which expects `model`,
`generation_params` and `prompt` so results are reproducible. Leave one out and you get a warning, and the
run records what was missing:

In [ ]:
incomplete = llmbenchmark.start_run("forgot-settings", model=CLAUDE_MODEL)
incomplete.finish()
incomplete.info.lineage.incomplete

## Step 4: Errors

Real benchmarks hit rate limits and timeouts. Inside `track()` the exception is **recorded** in the record's
`error` field; by default it's **re-raised**; with `raise_errors=False` it's recorded and **swallowed**.

In [ ]:
def flaky_call(question):
    raise TimeoutError("provider timed out")

with llmbenchmark.start_run("errors-demo", mode="external") as run:
    for q in ["q1", "q2"]:
        with run.track(input=q, raise_errors=False) as t:
            t.response = flaky_call(q)

[(r.input, r.error) for r in run.records]

## Step 5: Benchmark two models over the dataset

One run per model, same dataset and prompt version, so the runs are directly comparable.

In [ ]:
models = {"claude": (CLAUDE_MODEL, ask_claude), "gpt": (OPENAI_MODEL, ask_gpt)}
runs = {}

for label, (model, ask) in models.items():
    with llmbenchmark.start_run(f"qa-{label}", model=model, generation_params=GEN_PARAMS,
                                prompt=prompt, dataset=dataset, tags=["tutorial"]) as run:
        for item in dataset:
            with run.track(item=item, raise_errors=False) as t:
                t.response = ask(prompt.render(item))
    runs[label] = run
    print(run)

In [ ]:
import pandas as pd

records = pd.concat([run.to_dataframe() for run in runs.values()], ignore_index=True)
records.groupby("model").agg(
    records=("id", "count"),
    errors=("error", lambda s: int(s.notna().sum())),
    mean_latency_ms=("latency_ms", "mean"),
    total_input_tokens=("input_tokens", "sum"),
    total_output_tokens=("output_tokens", "sum"),
).round(1)

## Step 6: A multi-turn conversation

Pass the whole conversation as `messages`. The record keeps every turn, takes the system prompt from the
system message, and uses the last user turn as its `input`. Multi-turn metrics (Step 8) score these.

In [ ]:
with llmbenchmark.start_run("conversation-demo", model=CLAUDE_MODEL, generation_params=GEN_PARAMS,
                            prompt=prompt) as convo_run:
    conversation = [
        {"role": "system", "content": "You are a friendly travel agent."},
        {"role": "user", "content": "I'd like to visit Paris."},
        {"role": "assistant", "content": "Lovely! When would you like to go?"},
        {"role": "user", "content": "In May, for a week."},
    ]
    with convo_run.track(messages=conversation, expected_output="A booking proposal for Paris in May") as t:
        t.output = "Great choice! Here are three hotels for a week in Paris in May..."   # or t.response = ...

c = convo_run.records[0]
print(c.kind, "| system:", c.system, "| input:", c.input, "| turns:", len(c.messages))

## Step 7: Trace an agent

An agent is judged on *how* it worked: which tools it called, in what order, with what results.
Wrap one task in `run.trace(...)`, and mark steps with `llmbenchmark.span(...)` or the `@llmbenchmark.observe` decorator.
Spans nest automatically (also across `async` code); for thread pools, wrap the function with `llmbenchmark.propagate`.

In [ ]:
KB = {"refund": "Refunds are accepted within 30 days with a receipt.",
      "shipping": "Standard shipping takes 3-5 business days."}

@llmbenchmark.observe(kind="tool")
def search_kb(topic):
    return KB.get(topic, "no article found")

@llmbenchmark.observe(kind="tool")
def lookup_order(order_id):
    return {"order_id": order_id, "status": "delivered", "days_ago": 12}

def support_agent(question, order_id=None):
    with llmbenchmark.span("plan", kind="llm") as s:
        s.response = ask_claude(f"Which knowledge-base topic answers: {question}?")
    article = search_kb("refund" if "refund" in question.lower() else "shipping")
    order = lookup_order(order_id) if order_id else None
    with llmbenchmark.span("answer", kind="llm") as s:
        s.response = ask_claude(f"Answer using: {article} {order}")
    return f"{article} Your order was delivered {order['days_ago']} days ago." if order else article

tasks = [
    {"input": "Can I get a refund for order 1234?", "order_id": "1234", "expected_tools": ["search_kb", "lookup_order"]},
    {"input": "Can I get a refund?",                "order_id": None,   "expected_tools": ["search_kb", "lookup_order"]},
]

with llmbenchmark.start_run("agent-demo", model=CLAUDE_MODEL, generation_params=GEN_PARAMS, prompt=prompt) as agent_run:
    for task in tasks:
        with agent_run.trace(input=task["input"], expected_tools=task["expected_tools"]) as tr:
            tr.output = support_agent(task["input"], task["order_id"])

tr = agent_run.records[0]
print("tools called:", [t.name for t in tr.tools_called], "| tokens:", tr.input_tokens, "/", tr.output_tokens)
pd.DataFrame([{"span": s.name, "kind": s.kind, "parent": next((p.name for p in agent_run.spans[tr.trace_id] if p.id == s.parent_id), None)}
              for s in agent_run.spans[tr.trace_id]])

## Step 8: Evaluate with llmbenchmark (managed evaluation)

`evaluate_run(run, metrics=[...])` turns each record into a DeepEval test case, runs the metrics, and stores
the results as one **Evaluation**: a versioned scoring pass that records the metrics, thresholds, judge and SDK version.

### 8a. Free, deterministic metrics

- `ExactMatch`: is the output exactly the expected answer? Strict: `"7"` ≠ `"Seven"`. That strictness is why LLM judges exist.
- `ExpectedTools`: did the agent call every tool it was expected to?

In [ ]:
from llmbenchmark.eval import evaluate_run, ExactMatch, ExpectedTools

exact = {label: evaluate_run(run, metrics=[ExactMatch()]) for label, run in runs.items()}
display(pd.concat({label: report.summary() for label, report in exact.items()}, names=["run"]))

agent_report = evaluate_run(agent_run, metrics=[ExpectedTools()])
agent_report.to_dataframe()[["metric", "score", "passed", "reason"]]

### 8b. Required fields are checked before judging

A metric that needs a field the record doesn't have (here: `ExactMatch` needs `expected_output`) is not
sent to a judge. The pair is recorded with an `error`, or, with `strict=True`, the call refuses to run.

In [ ]:
with llmbenchmark.start_run("no-expected", model=CLAUDE_MODEL, generation_params=GEN_PARAMS, prompt=prompt) as bare:
    with bare.track(input="What is 2 + 2?") as t:
        t.output = "4"

print(evaluate_run(bare, metrics=[ExactMatch()]).scores[0].error)
try:
    evaluate_run(bare, metrics=[ExactMatch()], strict=True)
except ValueError as exc:
    print("strict:", exc)

### 8c. LLM-as-a-judge metrics (costs tokens)

The judge is **Claude** (`claude-sonnet-5-5`) by default. Every metric × record is at least one judge call,
so this is skipped without an Anthropic key. Multi-turn metrics (e.g. `ConversationCompletenessMetric`) only
score conversation records; trace metrics (e.g. `TaskCompletionMetric`) only score agent records.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric, ConversationCompletenessMetric, TaskCompletionMetric
from llmbenchmark.eval import judge, correctness

if HAS_ANTHROPIC:
    j = judge()
    judged = {label: evaluate_run(run, metrics=[AnswerRelevancyMetric(threshold=0.7, model=j), correctness(model=j)],
                                  max_concurrent=5)
              for label, run in runs.items()}
    display(pd.concat({label: r.summary() for label, r in judged.items()}, names=["run"]).round(3))
    display(evaluate_run(convo_run, metrics=[ConversationCompletenessMetric(model=j)]).summary())
    display(evaluate_run(agent_run, metrics=[TaskCompletionMetric(model=j)]).summary())
else:
    judged = {}
    print("Skipped: set ANTHROPIC_API_KEY to run the Claude judge.")

## Step 9: Bring your own evaluation

Some teams evaluate with their own methods and just upload the results. Use `mode="external"` and an
**evaluation** named after your evaluator. Each score belongs to a record; run-level numbers (e.g. accuracy
over the whole set) go in with `ev.metric(...)`. Leaving the `with` block **finalizes** (freezes) the evaluation.

In [ ]:
my_results = [  # e.g. produced by your own scoring script
    {"id": "france", "prompt": "What is the capital of France?", "answer": "Paris", "gold": "Paris", "correct": 1},
    {"id": "gold",   "prompt": "What is the chemical symbol for gold?", "answer": "Ag", "gold": "Au", "correct": 0},
    {"id": "mars",   "prompt": "What planet is known as the Red Planet?", "answer": "Mars", "gold": "Mars", "correct": 1},
]

with llmbenchmark.start_run("team-x-results", mode="external", model="team-x-model") as byo_run:
    with byo_run.evaluation("team-x-rubric", version="1.0", metrics={"accuracy": {"threshold": 1}}) as ev:
        for row in my_results:
            rec = byo_run.log(input=row["prompt"], output=row["answer"], expected_output=row["gold"])
            ev.score(rec, "accuracy", row["correct"], reason="graded by team-x script")
        ev.metric("accuracy", sum(r["correct"] for r in my_results) / len(my_results), n=len(my_results))

print(ev)
ev.summary()

Or in one call from a DataFrame (or list of dicts):

In [ ]:
with llmbenchmark.start_run("team-x-bulk", mode="external", model="team-x-model") as bulk_run:
    bulk_ev = bulk_run.log_results(
        pd.DataFrame(my_results), evaluator="team-x-rubric", version="1.0",
        input_col="prompt", output_col="answer", expected_col="gold", item_id_col="id",
        scores={"accuracy": "correct"}, metrics={"accuracy": {"threshold": 1}}, aggregates={"accuracy": 2 / 3},
    )
httpx.get(f"{BACKEND}/v1/evaluations/{bulk_ev.id}/summary").json()["metrics"]

## Step 10: Version control — re-evaluate without overwriting

Scores are never edited in place. To re-score a run (new rubric, new judge, bug fix), create a **new** evaluation,
optionally marking it as superseding the old one. Both stay readable; the old one points to its replacement.

In [ ]:
v1 = exact["gpt"]
v2 = evaluate_run(runs["gpt"], metrics=[ExactMatch(case_sensitive=True)], supersedes=v1)

versions = httpx.get(f"{BACKEND}/v1/runs/{runs['gpt'].id}/evaluations").json()
pd.DataFrame(versions)[["id", "evaluator_version", "config_hash", "status", "superseded_by"]]

A finalized evaluation and a finished run are **frozen**: the backend refuses changes (HTTP 409) but still accepts
late retries of data created before they closed, so an outage never loses results. Every state change is in the audit log:

In [ ]:
pd.DataFrame(httpx.get(f"{BACKEND}/v1/audit", params={"limit": 8}).json())[["at", "actor", "action", "target_type"]]

## Step 11: Compare runs and set a baseline

`compare(base, head)` matches records by dataset item id and reports, per metric, how scores moved and which
items **regressed** (passed before, fail now). It warns when the two sides aren't like-for-like (different dataset
version or scoring config).

In [ ]:
diff = llmbenchmark.compare(base=exact["claude"], head=exact["gpt"])
print(diff)
print("warnings:", diff.warnings)
print("differences:", list(diff.differences))
display(diff.summary())
pd.DataFrame(diff.regressions)[["key", "input", "base_score", "head_score"]]

In [ ]:
llmbenchmark.set_baseline(exact["claude"])          # the run new candidates must beat
print(llmbenchmark.get_baseline())
llmbenchmark.compare(head=exact["gpt"]).summary()   # no base: compares against the baseline

## Step 12: Read results back

Everything is stored server-side, so you can analyze it later or from another notebook. To evaluate an old
run in a fresh session, pass its id to `evaluate_run`.

In [ ]:
all_runs = httpx.get(f"{BACKEND}/v1/runs", params={"project": "tutorial"}).json()
pd.DataFrame(all_runs)[["name", "mode", "model", "status", "started_at"]].head(10)

In [ ]:
claude_run_id = runs["claude"].id
report = evaluate_run(claude_run_id, metrics=[ExactMatch()], upload=False)   # records fetched from the backend
report.summary()

## Step 13: When the backend is down

The SDK **never breaks your notebook** because of the backend. It retries, then saves the payloads to
`.llmbenchmark/<run_id>.jsonl` and logs a warning. `llmbenchmark.replay()` sends them later.

In [ ]:
# Point the SDK at a port where nothing is listening.
llmbenchmark.init(endpoint="http://localhost:9", project="tutorial", max_retries=1)

with llmbenchmark.start_run("offline-demo", mode="external") as offline_run:
    offline_run.log(input="still recorded?", output="yes")   # no exception, just warnings

fallback_file = config.fallback_dir / f"{offline_run.id}.jsonl"
print("saved locally:", fallback_file, "exists:", fallback_file.exists())

In [ ]:
# Backend reachable again: point back at it and replay.
llmbenchmark.init(endpoint=BACKEND, project="tutorial")
sent, failed = llmbenchmark.replay(fallback_file)
print(f"replayed: {sent} sent, {failed} failed")
print("on server now:", httpx.get(f"{BACKEND}/v1/runs/{offline_run.id}").json()["status"])

## Cheat sheet

```python
import llmbenchmark
llmbenchmark.init(endpoint=..., project=...)                    # once per notebook

prompt  = llmbenchmark.Prompt("qa", system=..., template="{input}", version="v1")
dataset = llmbenchmark.Dataset("qa", [{"id": ..., "input": ..., "expected_output": ...}], version=...)

with llmbenchmark.start_run("name", model=..., generation_params={...}, prompt=prompt, dataset=dataset) as run:
    with run.track(item=item) as t:                  # or input=..., system=..., messages=[...]
        t.response = client.messages.create(...)
    with run.trace(input=task, expected_tools=[...]) as tr:     # agents
        with llmbenchmark.span("step", kind="llm"): ...
        tr.output = answer

from llmbenchmark.eval import evaluate_run, judge, correctness, ExactMatch, ExpectedTools
report = evaluate_run(run, metrics=[ExactMatch(), correctness(model=judge())])

with llmbenchmark.start_run("mine", mode="external") as run:     # your own scores
    with run.evaluation("my-rubric", version="1") as ev:
        ev.score(run.log(input=q, output=a), "accuracy", 1.0)

llmbenchmark.compare(base=old_report, head=report); llmbenchmark.set_baseline(report)
llmbenchmark.replay(".llmbenchmark/<run_id>.jsonl")             # re-send after an outage
```

More background: `docs/CONCEPTS.md`.

In [ ]:
# Clean up: stop the backend if this notebook started it.
if server_process is not None:
    server_process.terminate()
    print("backend stopped")